In [2]:
import subprocess
import sys

# ---------------------------------------------------------
# Variables
# ---------------------------------------------------------
PROJECT_ID = "good-doctor-titan-389604"
REGION = "asia-southeast2"
AR_REPO = "ml-containers"
CLOUD_RUN_SERVICE_NAME = "stg-gd-recommendation"

# ---------------------------------------------------------
# 1. Check & Create Artifact Registry Repository
# ---------------------------------------------------------
print(f"Checking if Artifact Registry repository '{AR_REPO}' exists...")
repo_check = subprocess.run(
    [
        "gcloud", "artifacts", "repositories", "describe", AR_REPO,
        f"--location={REGION}",
        f"--project={PROJECT_ID}"
    ],
    capture_output=True,
    text=True
)

if repo_check.returncode != 0:
    print(f"Repository '{AR_REPO}' not found. Creating it now...")
    create_repo = subprocess.run(
        [
            "gcloud", "artifacts", "repositories", "create", AR_REPO,
            "--repository-format=docker",
            f"--location={REGION}",
            f"--project={PROJECT_ID}",
            "--description=ML model containers"
        ],
        text=True
    )
    if create_repo.returncode != 0:
        print("Error: Failed to create repository.")
        sys.exit(1)

print(f"Artifact Registry repository ready: {AR_REPO}")

# ---------------------------------------------------------
# 2. Build Container Image via Cloud Build
# ---------------------------------------------------------
ar_image = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{CLOUD_RUN_SERVICE_NAME}"

print(f"\nBuilding container image: {ar_image}:latest")
print("-" * 50)

build_process = subprocess.run(
    [
        "gcloud", "builds", "submit",
        f"--tag={ar_image}:latest",
        f"--project={PROJECT_ID}"
    ],
    text=True
)

if build_process.returncode != 0:
    print("Error: Cloud Build failed.")
    sys.exit(1)

# ---------------------------------------------------------
# 3. Deploy to Cloud Run
# ---------------------------------------------------------
print(f"\nDeploying {CLOUD_RUN_SERVICE_NAME} to Cloud Run...")
print("-" * 50)

deploy_process = subprocess.run(
    [
        "gcloud", "run", "deploy", CLOUD_RUN_SERVICE_NAME,
        f"--image={ar_image}:latest",
        f"--region={REGION}",
        f"--project={PROJECT_ID}",
        "--memory=1Gi",  # Increased to prevent OOM crashes on startup
        "--cpu=1",       # Increased to speed up ML model loading
        "--min-instances=0",
        "--max-instances=10",
        "--timeout=300",
        "--concurrency=80",
        "--allow-unauthenticated"
    ],
    text=True
)

if deploy_process.returncode != 0:
    print("Error: Cloud Run deployment failed.")
    sys.exit(1)

print(f"\nSuccessfully deployed {CLOUD_RUN_SERVICE_NAME}!")

Checking if Artifact Registry repository 'ml-containers' exists...
Artifact Registry repository ready: ml-containers

Building container image: asia-southeast2-docker.pkg.dev/good-doctor-titan-389604/ml-containers/stg-gd-recommendation:latest
--------------------------------------------------


Creating temporary archive of 13 file(s) totalling 138.9 KiB before compression.
Uploading tarball of [.] to [gs://good-doctor-titan-389604_cloudbuild/source/1773299521.80003-8a225f45d5294c55ad695214f9c615db.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/good-doctor-titan-389604/locations/global/builds/283ee990-d693-4af4-a4ef-dfcaf7cd04ea].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/283ee990-d693-4af4-a4ef-dfcaf7cd04ea?project=601326951594 ].
Waiting for build to complete. Polling interval: 1 second(s).


----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "283ee990-d693-4af4-a4ef-dfcaf7cd04ea"

FETCHSOURCE
Fetching storage object: gs://good-doctor-titan-389604_cloudbuild/source/1773299521.80003-8a225f45d5294c55ad695214f9c615db.tgz#1773299522874903
Copying gs://good-doctor-titan-389604_cloudbuild/source/1773299521.80003-8a225f45d5294c55ad695214f9c615db.tgz#1773299522874903...
/ [1 files][ 27.0 KiB/ 27.0 KiB]                                                
Operation completed over 1 objects/27.0 KiB.
BUILD
Already have image (with digest): gcr.io/cloud-builders/gcb-internal
Sending build context to Docker daemon  152.6kB
Step 1/10 : FROM python:3.10-slim
3.10-slim: Pulling from library/python
206356c42440: Already exists
aecf3e6b3e04: Pulling fs layer
44df7c9b0118: Pulling fs layer
7fe6b2798913: Pulling fs layer
7fe6b2798913: Download complete
aecf3e6b3e04: Download complete
44df7c9b0118: Verifying Checksum
44df7c9b0118: Download complete
aecf3

Deploying container to Cloud Run service [stg-gd-recommendation] in project [good-doctor-titan-389604] region [asia-southeast2]
Deploying new service...
Setting IAM Policy............warning
Creating Revision....................................................................................................................................................................................................................................................done
Routing traffic.....done
Completed with warnings:
  Setting IAM policy failed, try "gcloud beta run services add-iam-policy-binding --region=asia-southeast2 --member=allUsers --role=roles/run.invoker stg-gd-recommendation"
Service [stg-gd-recommendation] revision [stg-gd-recommendation-00001-dk6] has been deployed and is serving 100 percent of traffic.
Service URL: https://stg-gd-recommendation-601326951594.asia-southeast2.run.app



Successfully deployed stg-gd-recommendation!


In [3]:
# Get Cloud Run service URL
import subprocess
PROJECT_ID = "good-doctor-titan-389604"
REGION = "asia-southeast2"
AR_REPO = "ml-containers"
CLOUD_RUN_SERVICE_NAME = "stg-gd-recommendation"

result = subprocess.run(
    ["gcloud", "run", "services", "describe", CLOUD_RUN_SERVICE_NAME,
     "--platform=managed", f"--region={REGION}", f"--project={PROJECT_ID}",
     "--format=value(status.url)"],
    capture_output=True, text=True
)
CLOUD_RUN_URL = result.stdout.strip()

print(f"Cloud Run Service URL: {CLOUD_RUN_URL}")
print(f"\nEndpoints:")
print(f"  Health: GET {CLOUD_RUN_URL}/health")
print(f"  Predict: POST {CLOUD_RUN_URL}/predict")

Cloud Run Service URL: https://stg-gd-recommendation-47rvl6monq-et.a.run.app

Endpoints:
  Health: GET https://stg-gd-recommendation-47rvl6monq-et.a.run.app/health
  Predict: POST https://stg-gd-recommendation-47rvl6monq-et.a.run.app/predict


In [4]:
%%time
# Test Cloud Run deployment (with authentication)
import requests
import google.auth.transport.requests
import google.oauth2.id_token
#CLOUD_RUN_URL = 'https://gd-recommendation-601326951594.asia-southeast2.run.app'
# Get identity token for authenticated requests
CLOUD_RUN_URL = 'https://stg-gd-recommendation-47rvl6monq-et.a.run.app'
auth_req = google.auth.transport.requests.Request()

# If running with service account, get ID token
try:
    token = google.oauth2.id_token.fetch_id_token(auth_req, CLOUD_RUN_URL)
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    print("Using authenticated requests")
except Exception as e:
    # Fall back to unauthenticated if allUsers has invoker role
    headers = {"Content-Type": "application/json"}
    print(f"Using unauthenticated requests (auth error: {e})")
# Health check
response = requests.post(f"{CLOUD_RUN_URL}/recommendations?user_id=712224122&age=33&gender=M&diagnosis=R50.9&cart_products=23784&cart_products=23792&page=1&limit=100", headers=headers)
#print(f"Status: {response.status_code}, Response: {response.json()}")

Using authenticated requests
CPU times: user 34 ms, sys: 4.2 ms, total: 38.2 ms
Wall time: 133 ms


In [5]:
print(f"Status: {response.status_code}, Response: {response.json()}")

Status: 200, Response: {'data': [{'product_id': 51689, 'sku_name': 'Obat A 2', 'reason': 'Top Product'}, {'product_id': 51795, 'sku_name': 'Test Obat Subs A', 'reason': 'Top Product'}, {'product_id': 51796, 'sku_name': 'Test Obat Subs B', 'reason': 'Top Product'}, {'product_id': 51705, 'sku_name': 'Infalgin 500 mg Tab ', 'reason': 'Top Product'}, {'product_id': 65, 'sku_name': 'Laserin Madu Anak 60 ML Sirup', 'reason': 'Top Product'}, {'product_id': 61, 'sku_name': 'Holisticare Supreme Ester C Blackcurrant Eff', 'reason': 'Top Product'}, {'product_id': 54, 'sku_name': 'Rhinos Sr Capsul', 'reason': 'Top Product'}, {'product_id': 51707, 'sku_name': 'sanmol 120 mg per 5 ml syr ', 'reason': 'Top Product'}, {'product_id': 67, 'sku_name': 'Salonpas Jet Spray', 'reason': 'Top Product'}, {'product_id': 59, 'sku_name': 'Betadine Sol 15ml', 'reason': 'Top Product'}, {'product_id': 51797, 'sku_name': 'Test Obat Fahdii A', 'reason': 'Top Product'}, {'product_id': 51900, 'sku_name': 'Paracetamol Su

In [6]:
response = requests.get(f"{CLOUD_RUN_URL}/sbp_recommendations", headers=headers)
response.json()

{'data': [{'product_id': 52141,
   'sku_name': 'SBP Recommendation 1',
   'score': 999,
   'reason': 'Promoted Product'},
  {'product_id': 52142,
   'sku_name': 'SBP Recommendation 2',
   'score': 999,
   'reason': 'Promoted Product'},
  {'product_id': 52143,
   'sku_name': 'SBP Recommendation 3',
   'score': 999,
   'reason': 'Promoted Product'}],
 'meta': {'total_items': 3,
  'page': 1,
  'limit': 20,
  'total_pages': 1,
  'has_next': False,
  'has_prev': False}}

In [1]:
!gcloud artifacts docker images delete asia-southeast2-docker.pkg.dev/good-doctor-titan-389604/ml-containers/stg-gd-recommendation  --quiet
!gcloud run services delete stg-gd-recommendation --project=good-doctor-titan-389604 --region=asia-southeast2  --quiet

Delete request issued.
Waiting for operation [projects/good-doctor-titan-389604/locations/asia-southea
st2/operations/ef07ee01-d29c-48c3-b271-5927a621f187] to complete...done.       
Deleting [stg-gd-recommendation]...done.                                       
Deleted service [stg-gd-recommendation].


In [57]:
!curl -H "Authorization: Bearer $(gcloud auth print-identity-token)" \
     https://gd-recommendation-47rvl6monq-et.a.run.app/sbp_recommendations/


<html><head>
<meta http-equiv="content-type" content="text/html;charset=utf-8">
<title>404 Page not found</title>
</head>
<body text=#000000 bgcolor=#ffffff>
<h1>Error: Page not found</h1>
<h2>The requested URL was not found on this server.</h2>
<h2></h2>
</body></html>


In [58]:
CLOUD_RUN_URL

'https://stg-gd-recommendation-47rvl6monq-et.a.run.app'